# 05 - Análise espacial
Responde: quais regiões de Fortaleza concentram mais acidentes?

A base não tem bairro, então usamos duas formas:
1. ranking de logradouros e cruzamentos
2. grade (grid) sobre o mapa, contando acidentes por célula

Se der pra pegar o shapefile de bairros de Fortaleza (no mesmo portal de dados abertos), dá pra fazer também um `sjoin` por bairro. Fica como extra no final.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # pra achar a pasta src

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

import folium
from folium.plugins import HeatMap

In [ ]:
df = gpd.read_parquet(ARQ_LIMPO)

## Ruas com mais acidentes

In [ ]:
top_ruas = pd.concat([df["LOG1"], df["LOG2"]]).value_counts().head(15)
ax = top_ruas.plot(kind="barh", figsize=(8, 5), color="steelblue")
ax.invert_yaxis(); ax.set_title("Logradouros com mais acidentes")
plt.tight_layout(); plt.savefig(FIGURAS / "top_logradouros.png", dpi=150); plt.show()

In [ ]:
df["CRUZAMENTO"] = df["LOG1"] + " x " + df["LOG2"]
df["CRUZAMENTO"].value_counts().head(15)

## Mapa de pontos

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
df.plot(ax=ax, markersize=1, alpha=0.2, color="crimson")
ax.set_title("Acidentes em Fortaleza 2015-2024"); ax.set_axis_off()
plt.savefig(FIGURAS / "mapa_pontos.png", dpi=150); plt.show()

## Grade: contagem por célula

In [ ]:
# projeta pra metros (SIRGAS 2000 / UTM 24S) pra fazer células de tamanho fixo
dfm = df.to_crs(31984)
tam = 500  # metros

dfm["cx"] = (dfm.geometry.x // tam).astype(int)
dfm["cy"] = (dfm.geometry.y // tam).astype(int)

grade = (dfm.groupby(["cx", "cy"])
            .agg(acidentes=("ANO", "size"), mortos=("MORTOS", "sum"), feridos=("FERIDOS", "sum"))
            .reset_index())
grade["geometry"] = gpd.GeoSeries.from_xy(grade["cx"] * tam + tam / 2, grade["cy"] * tam + tam / 2, crs=31984) \
                       .buffer(tam / 2, cap_style=3)
grade = gpd.GeoDataFrame(grade, geometry="geometry", crs=31984)
grade.sort_values("acidentes", ascending=False).head(10)[["acidentes", "mortos", "feridos"]]

In [ ]:
ax = grade.plot(column="acidentes", cmap="YlOrRd", legend=True, figsize=(9, 9), scheme="quantiles", k=7)
ax.set_title(f"Acidentes por célula de {tam}m"); ax.set_axis_off()
plt.savefig(FIGURAS / "mapa_grade.png", dpi=150); plt.show()

## Mapa de calor interativo

In [ ]:
centro = [df["LATITUDE"].mean(), df["LONGITUDE"].mean()]
m = folium.Map(location=centro, zoom_start=12, tiles="cartodbpositron")
HeatMap(df[["LATITUDE", "LONGITUDE"]].sample(min(40000, len(df)), random_state=1).values.tolist(), radius=8, blur=10).add_to(m)
m.save(FIGURAS / "mapa_calor.html")
m

## Extra: por bairro
Baixar o shapefile de bairros, colocar em `data/raw/bairros/` e rodar:
```python
bairros = gpd.read_file(RAW / "bairros" / "bairros.shp").to_crs(df.crs)
j = gpd.sjoin(df, bairros, how="left", predicate="within")
j["NOME_DO_BAIRRO"].value_counts().head(15)
```
(o nome da coluna depende do arquivo)

In [ ]:
grade.to_parquet(PROCESSED / "05_grade.parquet")

## Conclusões deste notebook
(preencher)